<img src="https://raw.githubusercontent.com/PrimeReSolutions/python_actuarios/main/datos/img/logo_curso.png" width="450">

<p style="font-family: Arial; color: navy; text-align: center; font-size: 13px; letter-spacing: 1px; text-transform: uppercase; margin-bottom: 0;">
WSP — Python aplicado a modelos actuariales
</p>

<h1 style="background-color:#0070C0; color:white; text-align:center; font-family:Arial; padding:18px 0; border-radius:6px; margin-top:6px;">
Sesión 5 — Simulación y distribuciones de pérdida
</h1>

<div style="outline: 2px solid #EFB400; color:#000000; font-family: Arial; padding: 14px 18px; border-radius: 6px; margin-top: 14px;">
<h3 style="margin-top:0;">🎯 Objetivos</h3>
<ul>
<li>Generar variables aleatorias con <code>numpy.random</code> (uniforme, normal, Poisson, exponencial, gamma), retomando los arrays de NumPy que viste en la <b>Sesión 1</b>.</li>
<li>Ajustar y calibrar distribuciones de frecuencia y severidad con <code>scipy.stats</code>.</li>
<li>Evaluar la bondad de ajuste con las pruebas Chi-cuadrado y Kolmogorov-Smirnov.</li>
<li>Simular pérdidas agregadas con el método <b>Monte Carlo</b> (frecuencia x severidad) y estimar el riesgo de prima con VaR/TVaR.</li>
<li>Sentar las bases del riesgo de prima que retomarás en la <b>Sesión 6</b> para calcular el ratio de solvencia.</li>
</ul>
</div>

<div style="outline: 2px solid #0070C0; color:#000000; font-family: Arial; padding: 14px 18px; border-radius: 6px; margin-top: 12px;">
<h3 style="margin-top:0;">✍️ Cómo usar este notebook</h3>
<p>Los huecos que debes completar están marcados con <code>***</code>. Reemplázalos por el código o valor correspondiente y ejecuta la celda. El notebook <b>solucionario</b> (<code>solucionarios/sesion_05_simulacion_solucionario.ipynb</code>) tiene la respuesta completa de cada <code>***</code>, con la misma estructura de celdas que este notebook. Ejecuta las celdas en orden: varias reutilizan variables definidas en celdas anteriores.</p>
</div>


### ⚙️ Celda de arranque

Si trabajas en **Google Colab**, ejecuta esta celda **antes que cualquier otra**: descarga los datos del curso desde GitHub y deja el notebook listo para leerlos. Vuelve a ejecutarla cada vez que Colab reinicie el entorno. En tu PC (instalación local) no hace nada.

In [ ]:
# ⚙️ Celda de arranque: prepara el entorno en Google Colab (en tu PC no hace nada)
import os, sys

if "google.colab" in sys.modules:
    if not os.path.exists("/content/python_actuarios"):
        !git clone -q --depth 1 https://github.com/PrimeReSolutions/python_actuarios.git /content/python_actuarios
    %cd /content/python_actuarios/notebooks

In [ ]:
import pandas as pd
import numpy as np
import matplotlib as pl
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, FuncFormatter
import warnings
from scipy import stats

warnings.filterwarnings("ignore")

# Opciones de visualización
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:,.2f}".format)


print("pandas: " + pd.__version__)
print("numpy: " + np.__version__)

# 🧩 `np.random`

## 🎯 Objetivo
Familiarizarse con las principales funciones del módulo `numpy.random`,  
que permite **generar números aleatorios** y **simular distribuciones**.

---

## 💡 Idea
Montecarlo se apoya en la capacidad de generar valores aleatorios **de una distribución específica**:  
Uniforme, Normal, Poisson, Gamma, Exponencial, etc.

El submódulo `np.random` incluye funciones listas para cada una de ellas.

---

## 🧮 Estructura general
```python
np.random.<nombre_de_la_distribucion>(parametros, size=n)

> 🔗 **Conexión:** Los arrays de NumPy que usamos aquí (`np.random`) los conociste en la Sesión 1 (Fundamentos); ahora los usamos para generar variables aleatorias.




> Distribución Uniforme y Normal



In [ ]:
# Simulaciones
x_uni = np.random.uniform(low=***, high=***, size=***)
x_norm = np.random.normal(loc=***, scale=***, size=***)

# Visualización
plt.figure(figsize=(***,***))
plt.subplot(***,***,***)
plt.hist(***, bins=***, color="***", density=***)
plt.title("Uniforme(0,1)")
plt.subplot(***,***,***)
plt.hist(***, bins=***, color="***", density=***)
plt.title("Normal(0,1)")
plt.show()

> Distribución Poisson y Exponencial

In [ ]:
# Poisson: número de siniestros

x_pois = np.random.***(lam=***, size=***)

# Exponencial: tiempo entre siniestros

x_exp = np.random.***(scale=***, size=***)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.hist(x_pois, bins=range(0,15), density=True, color="orange")
plt.title("Poisson(λ=3)")
plt.subplot(1,2,2)
plt.hist(x_exp, bins=30, density=True, color="green")
plt.title("Exponencial(θ=2)")
plt.show()



> **Reproducibilidad:** Semillas aleatorias




In [ ]:
np.random.***(***)
x = np.random.***(size=***)
print(x)


## 📊 Distribuciones típicas en modelos actuariales de frecuencia y severidad

## 🎯 Idea general

En la modelación de riesgos, diferenciamos entre:

- **Frecuencia** → cuántos siniestros ocurren en un periodo.  
  *Variable discreta (conteo)*.

- **Severidad** → cuánto cuesta cada siniestro.  
  *Variable continua y positiva*.

---

## 🧩 Descripción conceptual

### 🔹 Frecuencia
- Se modela con **distribuciones discretas** porque los siniestros se cuentan (0, 1, 2…).
- Si los eventos son *independientes* y *raros*, el modelo natural es **Poisson**.
- Si existe **sobredispersión** (la varianza > media), se usa **Binomial Negativa**, que agrega heterogeneidad.

### 🔹 Severidad
- Se modela con **distribuciones continuas positivas**, ya que los montos nunca son negativos.
- **Gamma** y **Lognormal** modelan severidades moderadas (colas ligeras).
- **Pareto** captura colas pesadas y es ideal cuando existen pérdidas muy grandes con baja probabilidad.

---

## 📘 Tabla resumen

| **Tipo** | **Distribución** | **Usos típicos** | **Forma / Propiedades** |
|:--|:--|:--|:--|
| **Frecuencia** | **Poisson(λ)** | Conteo de siniestros con media ≈ varianza | Discreta; simétrica en torno a λ para valores grandes
|  | **Binomial Negativa(r, p)** | Frecuencia con alta varianza (sobredispersión) | Discreta; varianza > media
|  | **Binomial(n, p)** | Casos con límite máximo de ocurrencias | Discreta acotada (0…n)
| **Severidad** | **Gamma(k, θ)** | Montos moderados, colas ligeras | Continua, asimétrica, solo >0
|  | **Lognormal(μ, σ)** | Montos con crecimiento multiplicativo | Continua, positiva, cola moderada
|  | **Pareto(α, xₘ)** | Montos con colas pesadas (pérdidas extremas) | Continua, cola muy gruesa


---

## Ejercicio 1:  **Los reclamos del mes**

**Enunciado:**

Una compañía de seguros quiere modelar el comportamiento mensual de los reclamos de su cartera de automóviles.

Del análisis histórico, se sabe lo siguiente:

- En promedio, se registran entre 4 y 6 reclamos por mes.

- El tiempo entre reclamos tiende a concentrarse alrededor de 1.5 días, aunque hay casos extremos de más de 10 días.

Con esta información:

1. Elige qué distribución aleatoria usarías para modelar el número de reclamos por mes.

2. Elige otra para modelar el tiempo entre reclamos.

3. Simula 10 000 escenarios y grafica ambas distribuciones.

4. Interpreta brevemente los resultados.

In [ ]:
N = 10000

# Frecuencia mensual
reclamos_mes = np.random.***(lam=***, size=N)

tiempo = np.random.***(scale=***, size=N)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.hist(reclamos_mes, bins=range(0,15), color="orange", density=True)
plt.title("Distribución del número de reclamos")

plt.subplot(1,2,2)
plt.hist(tiempo, bins=40, color="green", density=True)
plt.title("Distribución del tiempo entre reclamos")
plt.show()




> Otras funciones útiles



In [ ]:
# 3 filas × 2 columnas de valores uniformes [0, 1)

np.random.***(***, ***)

In [ ]:
# 100 valores de una distribución normal estándar N(0,1)
np.random.***(***)

In [ ]:
# 5 enteros aleatorios entre 0 y 9
np.random.***(***, ***, size=***)


In [ ]:
# 10 valores uniformes [0, 1)
np.random.***(size=***)


In [ ]:
# Selecciona 5 elementos de la lista [1, 2, 3] con reemplazo
np.random.***([***], size=***, replace=***)


# 🧩 Introducción a `scipy.stats`

## 🎯 Objetivo
Aprender a utilizar el módulo `scipy.stats` para:
- Simular distribuciones (`rvs`),
- Calcular probabilidades (`pdf`, `cdf`),
- Estimar parámetros (`fit`),
- y **evaluar el ajuste de distribuciones** a datos observados.


---

## 💡 Idea
Mientras `np.random` sirve para **generar valores aleatorios**,  
`scipy.stats` nos permite **trabajar con las funciones estadísticas completas** de cada distribución.

Cada distribución (Normal, Poisson, Gamma, Pareto, etc.) tiene las mismas funciones principales:

| Función | Descripción | Ejemplo |
|----------|-------------|---------|
| `rvs()` | Genera valores aleatorios | `stats.gamma.rvs(a=2, scale=3, size=1000)` |
| `pdf()` | Densidad (probabilidad puntual) | `stats.gamma.pdf(x, a=2, scale=3)` |
| `cdf()` | Probabilidad acumulada | `stats.gamma.cdf(x, a=2, scale=3)` |
| `ppf()` | Inversa de la CDF (percentil) | `stats.gamma.ppf(0.95, a=2, scale=3)` |
| `fit()` | Ajusta parámetros a datos observados | `stats.gamma.fit(datos)` |

---



> Explorando una distribución Gamma



In [ ]:
# Parámetros de la distribución
forma, escala = 2.5, 1200

# Generamos datos simulados
x = stats.***.***(a=***, scale=***, size=10000)

# Densidad teórica
xs = np.***(***, ***, ***)
pdf_teo = stats.***.***(***, a=***, scale=***)

plt.hist(x, bins=***, density=***, alpha=***, label="Datos simulados")
plt.plot(xs, pdf_teo, 'r-', lw=2, label="PDF teórica Gamma")
plt.title("Distribución Gamma — Simulación vs PDF teórica")
plt.legend()
plt.show()


In [ ]:
# Ajuste de parámetros gamma
forma_est, loc_est, escala_est = stats.gamma.***(***)

print(f"Forma estimada: {forma_est:.2f}")
print(f"Escala estimada: {escala_est:.2f}")
print(f"Locación (offset): {loc_est:.2f}")


In [ ]:
# Ejemplo: ¿qué probabilidad hay de que X sea menor a 4000?

valor = ***
prob = stats.***.***(***, a=***, loc=***, scale=***)
print(f"P(X ≤ {valor:,.0f}) = {prob:.3%}")

In [ ]:
# Ejemplo: ¿cuál es el valor que está en la posición 95% de las observaciones?

percentil = 0.95
valor_p95 = stats.***.***(***, a=***, loc=***, scale=***)
print(f"Valor en el percentil 95%: {valor_p95:,.2f}")

In [ ]:
xs = np.linspace(0, np.***(x, 99.5), 300)

cdf_teo = stats.***.***(xs, a=***, loc=***, scale=***)

plt.figure(figsize=(***,***))

# --- Panel A: CDF ---
plt.subplot(1,2,1)
plt.plot(***, ***, 'b-', lw=***, label='CDF teórica')
plt.axvline(valor, color='gray', linestyle='--')
plt.axhline(prob, color='gray', linestyle='--')
plt.title("Función de distribución acumulada (CDF)")
plt.xlabel("Valor de X")
plt.ylabel("Probabilidad acumulada")
plt.legend()
plt.grid(alpha=0.3)

# --- Panel B: PPF ---
p_grid = np.linspace(0.01, 0.99, 200)
ppf_teo = stats.gamma.ppf(p_grid, a=forma_est, loc=loc_est, scale=escala_est)
plt.subplot(1,2,2)
plt.plot(p_grid, ppf_teo, 'r-', lw=2, label='PPF teórica (percentiles)')
plt.axhline(valor_p95, color='gray', linestyle='--')
plt.axvline(percentil, color='gray', linestyle='--')
plt.title("Función inversa (PPF)")
plt.xlabel("Probabilidad acumulada")
plt.ylabel("Valor correspondiente de X")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Pruebas de hipótesis y calibración

| Función             | Descripción                             | Ejemplo                                                    |
| ------------------- | --------------------------------------- | ---------------------------------------------------------- |
| `stats.ttest_ind()` | Prueba t para medias de dos grupos      | comparar siniestralidad entre dos ramos                    |
| `stats.chisquare()` | Prueba de bondad de ajuste (categorías) | verificar distribución de frecuencia observada vs esperada |
| `stats.ks_2samp()`  | Kolmogorov–Smirnov para dos muestras    | comparar dos distribuciones empíricas                      |
| `stats.kstest()`    | Kolmogorov–Smirnov univariada           | evaluar si los datos siguen una distribución teórica       |
| `stats.shapiro()`   | Test de normalidad (Shapiro–Wilk)       | verificar si los residuos son normales                     |
| `stats.anderson()`  | Anderson–Darling                        | prueba más potente para normalidad o colas pesadas         |


## Prueba Chi-cuadrado — stats.chisquare()

In [ ]:
# Frecuencias observadas (ej. número de siniestros por rango)
observed = np.array([18, 22, 20, 25, 15])

# Frecuencias esperadas (modelo teórico o histórico)
expected = np.array([20, 20, 20, 20, 20])

# Prueba Chi-cuadrado
chi2_stat, p_value = stats.chisquare(f_obs=***, f_exp=***)

print(f"Chi² estadístico: {chi2_stat:.3f}")
print(f"p-valor: {p_value:.3f}")

### 📊 Interpretación de resultados — Prueba Chi-cuadrado (`stats.chisquare`)

**🔹 p-valor = 0.575**

---

#### 🧮 Interpretación del p-valor
El p-valor representa la probabilidad de observar una diferencia tan grande (o mayor) entre las frecuencias observadas y las esperadas **si el modelo teórico fuera correcto**.  
En este caso, un p-valor de **0.575** indica que hay un **57.5 % de probabilidad** de que la discrepancia se deba simplemente al azar.

---

#### ⚖️ Decisión estadística
Usando un nivel de significancia habitual de α = 0.05:

- Como **0.575 > 0.05**, **no se rechaza la hipótesis nula (H₀)**.  
- Esto significa que **no hay evidencia estadísticamente significativa** para afirmar que las frecuencias observadas difieren del modelo teórico.

---

In [ ]:
plt.bar(range(len(***)), ***, alpha=0.6, label='Observados')
plt.bar(range(len(***)), ***, alpha=0.6, label='Esperados')
plt.legend()
plt.xlabel("Categorías / Rangos")
plt.ylabel("Frecuencia")
plt.title(f"Chi² = {chi2_stat:.2f}, p = {p_value:.3f}")
plt.show()


#### 💼 Interpretación actuarial
Las frecuencias observadas por rango de siniestros **no difieren significativamente** del patrón esperado.  
Por tanto, el modelo de frecuencia (por ejemplo, una distribución Poisson o el histórico usado como referencia) **describe adecuadamente el comportamiento de los siniestros observados**.  

En otras palabras, **el ajuste del modelo es aceptable desde el punto de vista actuarial**.

## Prueba Kolmogorov–Smirnov — stats.kstest()

In [ ]:
# Generamos datos simulados (por ejemplo, severidades)
data = stats.***.***(a=***, scale=***, size=***, random_state=***)

# Probamos si siguen una Gamma(a=2, scale=3)
ks_stat, p_value = stats.***(***, '***', args=(***, ***, ***))

print(f"KS estadístico: {ks_stat:.3f}")
print(f"p-valor: {p_value:.3f}")

### 📊 Interpretación de resultados — Prueba Kolmogorov–Smirnov (`stats.kstest`)

**🔹 p-valor = 0.341**

---

#### 🧮 Interpretación del p-valor
El p-valor indica la probabilidad de obtener una diferencia igual o mayor entre la distribución empírica de los datos y la distribución teórica especificada (en este caso, una Gamma con a = 2 y scale = 3), **si el modelo teórico fuera correcto**.  
Un p-valor de **0.341** significa que existe un **34.1 % de probabilidad** de observar una discrepancia como la actual solo por azar.

---

#### ⚖️ Decisión estadística
Usando un nivel de significancia α = 0.05:

- Como **0.341 > 0.05**, **no se rechaza la hipótesis nula (H₀)**.  
- No hay evidencia estadísticamente significativa para afirmar que los datos provienen de una distribución distinta a la Gamma(2, 3).

---

In [ ]:
# Ordenar datos para la CDF empírica
data_sorted = np.***(data)
cdf_empirical = np.***(1, len(***)+1) / len(***)

# CDF teórica Gamma(2,3)
cdf_theoretical = stats.gamma.***(***, a=2, scale=3)

# --- Plot ---
plt.figure(figsize=(6,4))
plt.plot(data_sorted, cdf_empirical, label='CDF empírica', lw=2)
plt.plot(data_sorted, cdf_theoretical, '--', label='CDF teórica (Gamma(2,3))', lw=2)
plt.xlabel('Severidad')
plt.ylabel('Probabilidad acumulada')
plt.title('Kolmogorov–Smirnov: CDF empírica vs teórica')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

#### 💼 Interpretación actuarial
Las severidades simuladas **no difieren significativamente** del modelo teórico Gamma(2, 3).  
Esto sugiere que la distribución Gamma **ajusta adecuadamente los datos observados**.



> Como se veria un ajuste malo?




In [ ]:
data = stats.***.rvs(s=0.5, scale=3, size=1000, random_state=42)

ks_stat, p_value = stats.kstest(data, 'gamma', args=(2, 0, 3))
print(f"KS estadístico: {ks_stat:.3f}")
print(f"p-valor: {p_value:.3f}")

In [ ]:

data_sorted = np.sort(data)
cdf_empirical = np.arange(1, len(data_sorted)+1) / len(data_sorted)

cdf_theoretical = stats.gamma.cdf(data_sorted, a=2, scale=3)

# --- Plot ---
plt.figure(figsize=(6,4))
plt.plot(data_sorted, cdf_empirical, label='CDF empírica', lw=2)
plt.plot(data_sorted, cdf_theoretical, '--', label='CDF teórica (Gamma(2,3))', lw=2)
plt.xlabel('Severidad')
plt.ylabel('Probabilidad acumulada')
plt.title('Kolmogorov–Smirnov: CDF empírica vs teórica')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 🧩 Introducción general a la simulación Montecarlo

## 🎯 Objetivo
Comprender la lógica detrás de una simulación Montecarlo:  
> “Usar números aleatorios para aproximar un resultado que sería muy difícil de calcular de forma exacta”.

---

## 💡 Idea
En seguros, muchas variables son **inciertas** (por ejemplo, cuántos siniestros habrá o cuánto costarán).  
Montecarlo nos permite **simular muchos escenarios posibles** y estimar:
- Promedios esperados  
- Percentiles (riesgo extremo)  
- Distribuciones completas de resultados

La idea básica es:

1. Generar **valores aleatorios** de las variables.  
2. Calcular el **resultado** en cada escenario.  
3. Repetir **muchas veces**.  
4. Analizar los resultados (media, desviación, percentiles, etc.).

---

## 🧮 Ejemplo general
Estimaremos el valor esperado de una variable aleatoria difícil de calcular directamente.  
Supón que queremos estimar la probabilidad de que la **suma de tres números aleatorios entre 0 y 1 sea mayor que 2**.

---

In [ ]:
# Número de simulaciones
N = ***

# Simulamos tres números aleatorios uniformes (entre 0 y 1)
x1 = np.random.***(N)
x2 = np.random.***(N)
x3 = np.random.***(N)

# Calculamos la suma y verificamos en cuántas veces supera 2
suma = x1 + x2 + x3
prob = np.***(suma > 2)

print(f"Probabilidad estimada de que la suma supere 2: {prob:.4f}")


## 🎲 Monte Carlo de pérdidas agregadas (frecuencia–severidad)

**Objetivo.** Dada una distribución de **frecuencia** y otra de **severidad**, simular la **pérdida agregada** por periodo:


$
S = \sum_{j=1}^{N} X_j,\quad N \sim \text{Frecuencia},\ X_j \sim \text{Severidad}
$

**Supuestos del ejemplo**
- Frecuencia: $N \sim \text{Poisson}(\lambda=3)$
- Severidad: $X \sim \text{Gamma}(k=2,\ \theta=1500)$
- Periodos simulados: 100,000




In [ ]:
# -----------------------
# 1) Parámetros
# -----------------------
rng = np.random.default_rng(42)
n_periodos = 10_000
lam = ***
k, theta = ***, ***

# -----------------------
# 2) Simulación
# -----------------------
# Frecuencia
N = rng.***(lam=***, size=***)

# Severidades
sev_all = stats.***.***(a=***, scale=***, size=N.sum(), random_state=***)


cuts = np.***(N)
S = np.***(n_periodos)
S[N > 0] = np.***.***(sev_all, np.r_[0, cuts[:-1]][N > 0])

# -----------------------
# 3) DataFrame a nivel PERIODO
# -----------------------
df_periodos = pd.DataFrame({
    "periodo": np.arange(1, n_periodos + 1),
    "num_siniestros": N,
    "monto_total": S
})

# -----------------------
# 4) DataFrame a nivel SINIESTRO
# -----------------------

periodos_rep = np.repeat(np.arange(1, n_periodos + 1), N)

df_siniestros = pd.DataFrame({
    "periodo": periodos_rep,
    "monto_siniestro": sev_all
})

In [ ]:

periodo_test = ***

suma_individual = df_siniestros.loc[df_siniestros["periodo"] == ***, "monto_siniestro"].sum()

suma_agrupada = df_periodos.loc[df_periodos["periodo"] == ***, "monto_total"].iloc[0]

print(f"Suma individual (periodo {periodo_test}): {suma_individual:,.2f}")
print(f"Suma en df_periodos  (periodo {periodo_test}): {suma_agrupada:,.2f}")
print(f"¿Coinciden?: {np.isclose(suma_individual, suma_agrupada)}")


In [ ]:
# -----------------------
# 3) Métricas
# -----------------------
EX = ***
VarX = ***
ES_teo = ***
VarS_teo = ***

ES_sim = S.***()
sdS_sim = S.***()
p95 = np.***(S, 95)
p99 = np.***(S, 99)

print(f"E[S] teórica   : {ES_teo:,.2f}")
print(f"E[S] simulada  : {ES_sim:,.2f}")
print(f"SD[S] simulada : {sdS_sim:,.2f}")
print(f"P95            : {p95:,.2f}")
print(f"P99            : {p99:,.2f}")

# -----------------------
# 4) Gráfico de la distribución de S
# -----------------------
plt.figure(figsize=(8,4.8))

plt.hist(S, bins=150, density=True, alpha=0.7, label="Agregada S")

for x, name in [(ES_sim, "Media simulada"), (p95, "P95"), (p99, "P99")]:
    plt.axvline(x, linestyle='--', linewidth=2, label=name)

plt.title("Monte Carlo Frecuencia–Severidad: distribución de la pérdida agregada S")
plt.xlabel("Pérdida agregada por periodo")
plt.ylabel("Densidad")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
prop_ceros = np.mean(S == 0)
print(f"Proporción de periodos sin siniestros: {prop_ceros:.3%}")
print(f"Teórico (Poisson λ={lam}): {np.exp(-lam):.3%}")


# 🧩 Estimación del riesgo de prima

## 🎯 Objetivo
Aplicar el método Montecarlo para simular **siniestros individuales** en una cartera:  
- La **frecuencia** de siniestros sigue una distribución *Poisson*.  
- La **severidad** (monto de cada siniestro) sigue una distribución *Pareto*.  
A partir de ello, estimaremos el **riesgo de prima**.

---

## 💡 Idea
El valor total de siniestros en un periodo puede verse como:


$S = X_1 + X_2 + \dots + X_N$


donde  
- $ N \sim \text{Poisson}(\lambda) $ = número de siniestros  
- $ X_i \sim \text{Pareto}(\alpha, x_m) $ = monto del siniestro *i*

Cada simulación Montecarlo generará un valor distinto de \( S \),  
y al repetir miles de veces podremos analizar su **distribución total**,  
el **valor esperado** y percentiles como el **p99 (riesgo)**.

---

> 🔗 **Conexión:** El riesgo de prima que estimamos aquí se retomará en la Sesión 6 para combinarlo con el riesgo de reserva y calcular el ratio de solvencia.


In [ ]:
rng = np.random.default_rng(7)   # semilla reproducible

In [ ]:
years = np.arange(2016, 2026)
lam_true = 4.5
alpha_true, xm_true = 2.2, 1_000.0

N_per_year = rng.poisson(lam=lam_true, size=len(years))

n_claims = int(N_per_year.sum())
sev_all = stats.pareto.rvs(alpha_true, loc=0, scale=xm_true, size=n_claims, random_state=rng)

year_rep = np.repeat(years, N_per_year)
df_claims = pd.DataFrame({
    "anio": year_rep,
    "monto": sev_all
})

df_claims


In [ ]:
df_year = (
    df_claims.groupby("***")["***"]
    .agg(num_siniestros="***", suma_siniestros="***")

)
df_year

In [ ]:
observed_counts = df_year["***"].***()
print("Frecuencias observadas:", observed_counts)


In [ ]:
observed_counts = df_year["***"].***()
n_years = len(observed_counts)

# Parámetro Poisson
lam_hat = observed_counts.***()


K = 10

vals = np.arange(0, K)

obs_main = np.array([(observed_counts == v).sum() for v in vals])
obs_tail = (observed_counts >= K).sum()
observed = np.append(obs_main, obs_tail)

# ---- Esperados teóricos ----
probs_main = stats.***.***(vals, lam_hat)
prob_tail  = 1 - stats.***.***(K-1, lam_hat)
expected = n_years * np.***(***, ***)

# Chi-cuadrado
chi2_stat, p_value = stats.***(f_obs=***, f_exp=***)
print("=== Chi-cuadrado Poisson ===")
print(f"Chi²: {chi2_stat:.3f}   p-valor: {p_value:.3f}")


In [ ]:
# Estimación por momentos
m = observed_counts.***()
v = observed_counts.***()

if v <= m:
    p_hat = 0.999
    n_hat = m * p_hat / (1 - p_hat)
else:
    p_hat = m / v
    n_hat = m**2 / (v - m)


probs_main_nb = stats.***.***(vals, n_hat, p_hat)
prob_tail_nb  = 1 - stats.***.***(K-1, n_hat, p_hat)
expected_nb = n_years * np.***(***, ***)

chi2_nb, p_nb = stats.chisquare(f_obs=***, f_exp=***)
print("=== Chi-cuadrado Neg. Binomial ===")
print(f"Chi²: {chi2_nb:.3f}   p-valor: {p_nb:.3f}")


In [ ]:

x = df_claims["monto"].to_numpy()

# --- LOGNORMAL:
s_logn, loc_logn, scale_logn = stats.***.***(***, floc=0)
ks_logn, p_logn = stats.***(x, 'lognorm', args=(***, ***, ***))

# --- PARETO :
a_par, loc_par, scale_par = stats.pareto.fit(x, floc=0)
ks_par, p_par = stats.kstest(x, 'pareto', args=(a_par, loc_par, scale_par))

print("== Severidad (K-S) ==")
print(f"Lognormal : KS={ks_logn:.3f}  p={p_logn:.3f}   params: s={s_logn:.4f}, loc={loc_logn:.2f}, scale={scale_logn:.2f}")
print(f"Pareto    : KS={ks_par:.3f}  p={p_par:.3f}     params: a={a_par:.4f}, loc={loc_par:.2f}, scale={scale_par:.2f}")

best_sev = "Pareto" if p_par >= p_logn else "Lognormal"
print(f"→ Modelo de severidad elegido: {best_sev}")

In [ ]:
lam_poisson = ***


def sample_frequency(size, lam=lam_poisson):
    """Frecuencia ~ Poisson(lam)"""
    return rng.poisson(lam=lam, size=size)

def sample_severity(n):

    if best_sev == "Pareto":
        return stats.pareto.***(***, ***, ***, size=n, random_state=rng)
    else:  # Lognormal
        return stats.lognorm.***(***, ***, ***, size=n, random_state=rng)

# Montecarlo

n_sim = 100_000
N_sim = sample_frequency(***, lam=***)
sev_sim = sample_severity(int(N_sim.sum()))

cuts = np.***(N_sim)
S = np.***(n_sim)
mask = N_sim > 0
S[mask] = np.***.***(sev_sim, np.r_[0, cuts[:-1]][mask])


ES, SD = S.***(), S.***()
p95, p99 = np.***(S, [95, 99])

print("== Monte Carlo – Pérdida agregada por año ==")
print(f"E[S]   : {ES:,.2f}")
print(f"SD[S]  : {SD:,.2f}")
print(f"P95    : {p95:,.2f}")
print(f"P99    : {p99:,.2f}")

plt.figure(figsize=(8,4.6))

plt.hist(S, bins=160, density=True, alpha=0.8, label="S (agregada)")
for x_val, lab in [(ES,"Media"), (p95,"P95"), (p99,"P99")]:
    plt.axvline(x_val, linestyle="--", linewidth=2, label=lab)
plt.xlim(0, p99 * 1.5)
plt.title("Monte Carlo – distribución de la pérdida agregada anual (Poisson λ)")
plt.xlabel("S por año")
plt.ylabel("Densidad")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
alpha = 0.99  # nivel de confianza
var = np.percentile(***, *** * 100)
tvar = S[S > var].***()
risk_premium = tvar - S.***()

print(f"Promedio: {S.mean():,.2f}")
print(f"VaR({alpha*100:.0f}%): {var:,.2f}")
print(f"TVaR({alpha*100:.0f}%): {tvar:,.2f}")
print(f"Riesgo de prima: {risk_premium:,.2f}")


## 🧪 Ejercicio: Frecuencia Binomial Negativa & Severidad Lognormal

**Objetivo.** Estimar el riesgo de prima:

1) Resumir **# siniestros** y **suma anual** por año de ocurrencia.
2) **Calibrar la frecuencia** con Chi-cuadrado comparando **Poisson vs Binomial Negativa**.
3) **Calibrar la severidad** con KS comparando **Lognormal vs Pareto**.
4) Elegir el **modelo ganador** en cada caso y realizar el **Monte Carlo** de la pérdida agregada anual \(S=\sum_{j=1}^{N}X_j\).
6) Reportar **E[S]**, **SD[S]**, **VaR**, **TVaR** y **Riesgo de prima = TVaR − E[S]**.


In [ ]:
years = np.arange(2016, 2026)

n_true, p_true = 3.0, 0.40

mu_true, sigma_true = 10.0, 0.60

N_per_year = stats.nbinom.rvs(n_true, p_true, size=len(years), random_state=rng)

n_claims = int(N_per_year.sum())
sev_all = stats.lognorm.rvs(s=sigma_true, scale=np.exp(mu_true), size=n_claims, random_state=rng)

year_rep = np.repeat(years, N_per_year)
df_claims = pd.DataFrame({"anio": year_rep, "monto": sev_all})
df_claims

In [ ]:
# Resumen por año: # siniestros y suma
df_year = (
    df_claims.groupby("***")["***"]
    .agg(num_siniestros="***", suma_siniestros="***")

)
df_year


In [ ]:
observed_counts = df_year["***"].***()
n_years = len(***)

# Poisson: lambda
lam_hat = observed_counts.***()

# NB: parámetros por momentos
m = observed_counts.***()
v = observed_counts.***()

if v <= m:
    p_hat = 0.999
    n_hat = m * p_hat / (1 - p_hat)
else:
    p_hat = m / v
    n_hat = m**2 / (v - m)



K = 10

vals = np.arange(0, K)
obs_main = np.array([(observed_counts == v).***() for v in vals])
obs_tail = (observed_counts >= K).***()
observed = np.***(obs_main, obs_tail)

# Esperados Poisson
probs_main_poi = stats.***.***(***, ***)
prob_tail_poi  = 1 - stats.***.***(K-1, ***)
expected_poi = n_years * np.append(***, ***)

# Esperados Neg. Binomial
probs_main_nb = stats.***.***(***, ***, ***)
prob_tail_nb  = 1 - stats.***.***(K-1, ***, ***)
expected_nb = n_years * np.append(***, ***)

chi2_poi, p_poi = stats.***(f_obs=***, f_exp=***)
chi2_nb,  p_nb  = stats.***(f_obs=***, f_exp=***)

print("== Frecuencia (Chi-cuadrado) ==")
print(f"Poisson   -> Chi²={chi2_poi:.3f}  p={p_poi:.3f}   (lambda_hat={lam_hat:.3f})")
print(f"NegBin    -> Chi²={chi2_nb:.3f}   p={p_nb:.3f}    (n_hat={n_hat:.3f}, p_hat={p_hat:.3f})")

best_freq = "NegBin" if p_nb >= p_poi else "Poisson"
print(f"→ Modelo frecuencia elegido: {best_freq}")


In [ ]:
x = df_claims["monto"].***()

# Lognormal
s_logn, loc_logn, scale_logn = stats.***.***(x, floc=0)
ks_logn, p_logn = stats.***(x, 'lognorm', args=(***, ***, ***))

# Pareto
a_par, loc_par, scale_par = stats.***.***(x, floc=0)
ks_par, p_par = stats.***(x, 'pareto', args=(***, ***, ***))

print("== Severidad (KS) ==")
print(f"Lognormal -> KS={ks_logn:.3f}  p={p_logn:.3f}   params: s={s_logn:.4f}, scale={scale_logn:.2f}")
print(f"Pareto    -> KS={ks_par:.3f}   p={p_par:.3f}    params: a={a_par:.4f}, scale={scale_par:.2f}")

best_sev = "Lognormal" if p_logn >= p_par else "Pareto"
print(f"→ Modelo severidad elegido: {best_sev}")


In [ ]:
def sample_frequency(size):
    if best_freq == "NegBin":
        return stats.nbinom.***(n_hat, p_hat, size=size, random_state=rng)
    else:
        return rng.***(lam=lam_hat, size=size)

def sample_severity(n):
    if best_sev == "Lognormal":
        return stats.lognorm.***(s_logn, loc_logn, scale_logn, size=n, random_state=rng)
    else:
        return stats.pareto.***(a_par, loc_par, scale_par, size=n, random_state=rng)

# Montecarlo
n_sim = 100_000
N_sim = sample_frequency(n_sim)
sev_sim = sample_severity(int(N_sim.sum()))

cuts = np.cumsum(N_sim)
S = np.zeros(n_sim)
mask = N_sim > 0
S[mask] = np.add.reduceat(sev_sim, np.r_[0, cuts[:-1]][mask])

# --- Métricas + VaR/TVaR + riesgo de prima ---
ES = S.***()
SD = S.***()
alpha = 0.99
VaR = np.***(S, alpha*100)
TVaR = S[S > VaR].***()
riesgo_prima = ***

print("== Monte Carlo – Pérdida agregada por año ==")
print(f"E[S]     : {ES:,.2f}")
print(f"SD[S]    : {SD:,.2f}")
print(f"VaR  {int(alpha*100)}% : {VaR:,.2f}")
print(f"TVaR {int(alpha*100)}% : {TVaR:,.2f}")
print(f"Riesgo de prima (TVaR - E[S]): {riesgo_prima:,.2f}")

plt.figure(figsize=(8,4.6))
plt.hist(S, bins=160, density=True, alpha=0.8, label="S (agregada)")
for x_val, lab in [(ES,"Media"), (VaR,f"VaR {int(alpha*100)}%"), (TVaR,f"TVaR {int(alpha*100)}%")]:
    plt.axvline(x_val, linestyle="--", linewidth=2, label=lab)
plt.xlim(0, np.percentile(S, 99)*1.2)  # recorte para ver mejor el cuerpo
plt.title("Monte Carlo – pérdida agregada anual (NB & Lognormal calibrados)")
plt.xlabel("S por año")
plt.ylabel("Densidad")
plt.legend()
plt.tight_layout()
plt.show()
